## Adapting HME100K to TrOCR Fine-Tuning

1. **Dataset Format:** HME100K has ~74k train PNG images of single handwritten math expr + `train_labels.txt` (format: `imgname.png	latex_string`). Images are real-world scans, diverse writers/symbols (249 classes). Test similar. No need for layout parsing—TrOCR treats as image-to-sequence (LaTeX).
2. **Preprocessing Fit:**
   - Load images + labels into HF `Dataset`.
   - Use `TrOCRProcessor` (ViT image_processor resizes/crops to 384x384px, normalizes 0-1; RoBERTa tokenizer for LaTeX). Handles RGB/gray; math PNGs fit.
   - Tokenize labels (max_len=512, pad/trunc); decoder learns LaTeX tokens (BPE handles \\, fractions, superscripts). Vocab fixed, but fine-tuning adapts embeddings.
3. **Model Choice:** `trocr-small-stage1` (ViT-small encoder + RoBERTa-small decoder, \~63M params). Low VRAM (\~8GB peak batch=16 fp16). Stage1 pretrained on synth text—good base before handwriting/math.
4. **Training Optimization for Kaggle:** fp16, gradient_checkpointing, batch=16, lr=5e-5, 10 epochs. CER eval metric. Save checkpoints.
5. **Challenges/Mitigation:** LaTeX specials ok via BPE; nested frac/superscripts learned from data. If OOM, reduce batch/gradient_acc. Post-process LaTeX if needed.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

!gcloud auth activate-service-account 644061078062-compute@developer.gserviceaccount.com --key-file=/content/drive/My\ Drive/COMP4471_Project/sanguine-casing-416907-cda60e59ac44.json --project=sanguine-casing-416907
!mkdir -p dataset/

!gcloud storage cp -r gs://temphmerichyz/HME100k/ dataset/

!ls

Activated service account credentials for: [644061078062-compute@developer.gserviceaccount.com]
Copying gs://temphmerichyz/HME100k/HME100k/readme.md.txt to file://dataset/HME100k/HME100k/readme.md.txt
Copying gs://temphmerichyz/HME100k/HME100k/subset/easy.json to file://dataset/HME100k/HME100k/subset/easy.json
Copying gs://temphmerichyz/HME100k/HME100k/subset/hard.json to file://dataset/HME100k/HME100k/subset/hard.json
Copying gs://temphmerichyz/HME100k/HME100k/subset/medium.json to file://dataset/HME100k/HME100k/subset/medium.json
Copying gs://temphmerichyz/HME100k/HME100k/test/test_images/test_10.jpg to file://dataset/HME100k/HME100k/test/test_images/test_10.jpg
Copying gs://temphmerichyz/HME100k/HME100k/test/test_images/test_1000.jpg to file://dataset/HME100k/HME100k/test/test_images/test_1000.jpg
Copying gs://temphmerichyz/HME100k/HME100k/test/test_images/test_10001.jpg to file://dataset/HME100k/HME100k/test/test_images/test_10001.jpg
Copying gs://temphmerichyz/HME100k/HME100k/test

In [ ]:
# Install dependencies (restart after)
!pip install datasets transformers accelerate evaluate jiwer
import os
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# user_secrets.set_secret("HF_TOKEN", "your_hf_token")  # For push_to_hub

In [ ]:
import os
from pathlib import Path
import pandas as pd
from PIL import Image
from datasets import Dataset, Image as HFImage
import numpy as np

# Paths (adjust if different Kaggle input path)
data_dir = Path('./dataset/HME100k/')
train_img_dir = data_dir / 'train' / 'train_images'
train_label_path = data_dir / 'train' / 'train_labels.txt'
test_img_dir = data_dir / 'test' / 'test_images'
test_label_path = data_dir / 'test' / 'test_labels.txt'

# Parse labels: img_name.png\tlatex
def parse_labels(label_file, img_dir):
    df = pd.read_csv(label_file, sep='\t', header=None, names=['img', 'latex'])
    df['img_path'] = (Path(img_dir) / df['img']).astype(str)
    return df

train_df = parse_labels(train_label_path, train_img_dir)
test_df = parse_labels(test_label_path, test_img_dir)

print(f'Train: {len(train_df)}, Test: {len(test_df)}')
print(train_df.head())

Train: 74502, Test: 24607
           img                                              latex  \
0  train_0.jpg  ( 2 ) 2 N a O H + C u S O _ { 4 } = N a _ { 2 ...   
1  train_2.jpg  \angle A C B = \angle A ^ { \prime } C B ^ { \...   
2  train_3.jpg  1 0 \times ( x + 5 ) = 1 3 \times ( \frac { 5 ...   
3  train_4.jpg  l _ { 2 } = O B = \frac { O A } { 4 } \times 3...   
4  train_6.jpg                         \angle A = 4 0 ^ { \circ }   

                                            img_path  
0  dataset/HME100k/HME100k/train/train_images/tra...  
1  dataset/HME100k/HME100k/train/train_images/tra...  
2  dataset/HME100k/HME100k/train/train_images/tra...  
3  dataset/HME100k/HME100k/train/train_images/tra...  
4  dataset/HME100k/HME100k/train/train_images/tra...  


In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
from datasets import Dataset

model_name = "microsoft/trocr-small-stage1"
processor = TrOCRProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name)

# Resize decoder for LaTeX (optional, but safe)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id

# HF Datasets
train_dataset = Dataset.from_pandas(train_df).cast_column("img_path", HFImage()).select(range(len(train_df) // 2))
test_dataset = Dataset.from_pandas(test_df).cast_column("img_path", HFImage())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of VisionEncoderDecoderModel were not initialized from the model checkpoint at microsoft/trocr-small-stage1 and are newly initialized: ['encoder.

In [ ]:
# Preprocessing function
max_target_length = 128  # Adjust for long LaTeX

def preprocess(examples):
    images = [img.convert('RGB') for img in examples['img_path']]
    pixel_values = processor(images, return_tensors='pt').pixel_values
    labels = processor.tokenizer(examples['latex'], padding='max_length', max_length=max_target_length, truncation=True).input_ids
    return {'pixel_values': pixel_values.squeeze(), 'labels': labels}

def preprocess_stream(examples):
    """Process in tiny batches to avoid OOM"""
    images = [img.convert('RGB') for img in examples['img_path']]
    pixel_values = processor(images, return_tensors='pt').pixel_values.squeeze()
    labels = processor.tokenizer(
        examples['latex'],
        padding='max_length',
        max_length=256,  # Reduced from 512
        truncation=True,
        return_tensors='pt'
    ).input_ids
    return {'pixel_values': pixel_values, 'labels': labels}

train_dataset = train_dataset.map(preprocess_stream, remove_columns=train_dataset.column_names, batched=True, batch_size=8)
test_dataset = test_dataset.map(preprocess_stream, remove_columns=test_dataset.column_names, batched=True, batch_size=8)

Map:   0%|          | 0/37251 [00:00<?, ? examples/s]

Map:   0%|          | 0/24607 [00:00<?, ? examples/s]

In [ ]:
import evaluate
import numpy as np
from transformers import DataCollatorForSeq2Seq
import torch

cer_metric = evaluate.load("cer")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # If preds are logits (model returned logits), turn to token ids:
    if preds is None:
        return {}
    if preds.ndim == 3:  # logits -> argmax across vocab
        preds = np.argmax(preds, axis=-1)

    # Replace -100 in labels with pad_token_id so tokenizer won't index -100
    labels = np.where(labels != -100, labels, processor.tokenizer.pad_token_id)

    decoded_preds = processor.tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = processor.tokenizer.batch_decode(labels, skip_special_tokens=True)

    cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)

    # Calculate accuracy (exact match)
    exact_matches = [1 if pred == label else 0 for pred, label in zip(decoded_preds, decoded_labels)]
    accuracy = sum(exact_matches) / len(exact_matches) if len(exact_matches) > 0 else 0.0

    return {"cer": cer, "accuracy": accuracy}

def custom_data_collator(batch):
    """Collate function to stack image pixel arrays and pad/mask label sequences.

    Expects each example to have keys:
        - 'pixel_values': nested list or numpy array (C,H,W)
        - 'labels': list[int]
    Returns a dict with torch tensors: 'pixel_values' and 'labels' (labels masked with -100).
    """
    # Stack pixel values (each entry is a numpy array or list)
    pixel_list = [np.array(example["pixel_values"]) for example in batch]
    pixel_values = torch.tensor(np.stack(pixel_list))

    # Pad labels to max length in batch and replace pad_token_id with -100
    label_tensors = [torch.tensor(example["labels"], dtype=torch.long) for example in batch]
    labels_padded = torch.nn.utils.rnn.pad_sequence(label_tensors, batch_first=True, padding_value=processor.tokenizer.pad_token_id)
    labels_padded = labels_padded.masked_fill(labels_padded == processor.tokenizer.pad_token_id, -100)

    return {"pixel_values": pixel_values, "labels": labels_padded}

data_collator = DataCollatorForSeq2Seq(tokenizer=processor.tokenizer, model=model, padding=True)

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import torch

output_dir = './models'
output_dir = output_dir + '/12000_10_epoch'
os.makedirs(output_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    group_by_length=False,  # Ignore IndexError
    #length_column_name="labels",
    per_device_train_batch_size=256,  # Adjust for VRAM
    per_device_eval_batch_size=256,
    metric_for_best_model="cer",
    greater_is_better=False,
    save_strategy="epoch",
    save_total_limit=3,
    num_train_epochs=20,
    fp16=True,
    predict_with_generate=True,
    logging_steps=10,
    report_to=[],  # Changed to an empty list to explicitly disable reporting
    gradient_checkpointing=True,  # Mem save
    learning_rate=5e-5,
    warmup_steps=500,
    push_to_hub=False
)



trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.feature_extractor,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset.select(range(30000)),
    eval_dataset=test_dataset.select(range(2000)),  # Subset for eval
    data_collator=custom_data_collator,
)

trainer.train()
trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/transformers/models/trocr/processing_trocr.py:139: FutureWarning: `feature_extractor` is deprecated and will be removed in v5. Use `image_processor` instead.
  warnings.warn(
/tmp/ipython-input-3244671237.py:31: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Step,Training Loss
10,2.540600
20,2.475400
30,2.488800
40,2.426300
50,2.427300
60,2.413900
70,2.326900
80,2.321900
90,2.294000
100,2.222600


{'eval_loss': 0.874703049659729,
 'eval_cer': 0.6071214392803598,
 'eval_runtime': 464.8105,
 'eval_samples_per_second': 4.303,
 'eval_steps_per_second': 0.017,
 'epoch': 10.0}

In [ ]:
# Save model
trainer.save_model()
processor.save_pretrained(output_dir)


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
Device set to use cuda:0


a + y 2


In [15]:
# Inference example
from transformers import pipeline
pipe = pipeline("image-to-text", model=output_dir, tokenizer=output_dir)
img = Image.open(next(train_img_dir.iterdir()))
print(pipe(img)[0]['generated_text'])

HFValidationError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/COMP_4471_Project/models/1000_10_epoch'. Use `repo_type` argument if needed.

In [ ]:
!cp -rf /content/models /content/drive/My\ Drive/COMP4471_Project/

In [ ]:
from google.colab import runtime
runtime.unassign()

## Next Steps
- Download `/kaggle/working/trocr-hme100k` to HF repo.\n
- Monitor CER drop; extend epochs if needed.\n
- For full eval: trainer.evaluate(test_dataset).\n
- Nested frac/supers: Inspect gen LaTeX quality.